# 01 — CU2 baseline: TF-IDF + Regresión logística

Antes de ajustar BETO (notebook 02), este notebook entrena un modelo
simple de referencia: TF-IDF (unigramas y bigramas) + regresión
logística. Sirve para comprobar que BETO de verdad aporta algo —la meta
del T1 (OE2) es que BETO supere a este baseline, no solo que llegue a
F1-macro ≥ 0,85.

Tiempo esperado: menos de 5 minutos incluso con el dataset completo (no
usa GPU).

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {{CARPETA_ML}}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): usa la carpeta ml/ del repo.
    CARPETA_ML = str(__import__("pathlib").Path.cwd())

sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

# --- Configuración: edita esto antes de correr en Colab ---
CARPETA_DRIVE = Path(CARPETA_ML) / "datos"  # train.csv/val.csv/test.csv (los crea 00_preparar_dataset.ipynb)
CARPETA_MODELOS = Path(CARPETA_ML) / "modelos" / "cu2-tfidf-logreg-v1"
MODO_RAPIDO = False  # True = 5 ejemplos/clase, 1 época — solo para probar que el notebook corre de punta a punta
SEMILLA = 42
# ------------------------------------------------------------

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODO_RAPIDO={MODO_RAPIDO} · dispositivo={DISPOSITIVO} · GPU disponible: {torch.cuda.is_available()}")
if DISPOSITIVO == "cpu":
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)


## 1. Cargar la partición (la que armó 00_preparar_dataset.ipynb)

In [ ]:
import pandas as pd

if MODO_RAPIDO:
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(CARPETA_DRIVE)


train = pd.read_csv(CARPETA_DRIVE / "train.csv")
val = pd.read_csv(CARPETA_DRIVE / "val.csv")
test = pd.read_csv(CARPETA_DRIVE / "test.csv")

if MODO_RAPIDO:
    train = train.groupby("categoria", group_keys=False).head(5)
    val = val.groupby("categoria", group_keys=False).head(2)
    test = test.groupby("categoria", group_keys=False).head(2)

print(f"train={len(train)} val={len(val)} test={len(test)}")
train.head()


## 2. Entrenar TF-IDF + regresión logística

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from mapeo_categorias import CATEGORIAS as CLASES  # las 21, en orden fijo — no solo las que aparezcan en train

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20_000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
pipeline.fit(train["texto_es"], train["categoria"])
print("Entrenado.")


## 3. Evaluar sobre el conjunto de PRUEBA

(No sobre validación — las métricas que importan para el informe y para
`modelos_ia` son las del conjunto de prueba, que el modelo nunca vio.)

In [ ]:
import json

from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

y_test = test["categoria"]
y_pred = pipeline.predict(test["texto_es"])

metricas = {
    "accuracy": float((y_pred == y_test).mean()),
    "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
    "f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
    "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
    # labels=CLASES a propósito: igual que en 02_cu2_beto, si el conjunto
    # de prueba no tiene ejemplos de alguna categoría el reporte debe
    # listarla igual con ceros, no reventar ni desaparecerla en silencio.
    "reporte_por_clase": classification_report(y_test, y_pred, labels=CLASES, output_dict=True, zero_division=0),
}
print(json.dumps({k: v for k, v in metricas.items() if k != "reporte_por_clase"}, indent=2, ensure_ascii=False))

with open(CARPETA_MODELOS / "metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)


## 4. Matriz de confusión

(No hay curvas de pérdida/accuracy por época para este baseline: la
regresión logística de scikit-learn no entrena por épocas como una red
neuronal, así que esa gráfica no aplica acá — sí aplica, y se genera, en
02_cu2_beto y 03_cu1_imagen.)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, xticks_rotation="vertical", ax=ax, colorbar=False
)
plt.tight_layout()
plt.savefig(CARPETA_MODELOS / "matriz_confusion.png", dpi=120)
plt.show()


## 5. Guardar el pipeline y el SQL para registrar el modelo

In [ ]:
joblib.dump(pipeline, CARPETA_MODELOS / "pipeline.joblib")

sql = f"""
UPDATE novastore.modelos_ia
   SET metricas = '{json.dumps({k: v for k, v in metricas.items() if k != "reporte_por_clase"})}'::jsonb,
       hiperparametros = hiperparametros || '{{"ngram_range": [1, 2], "max_features": 20000}}'::jsonb,
       estado = 'evaluado',
       entrenado_en = now(),
       ruta_artefacto = 'cu2-tfidf-logreg-v1/pipeline.joblib'
 WHERE codigo = 'cu2-tfidf-logreg-v1';
"""
(CARPETA_MODELOS / "registro_modelo.sql").write_text(sql, encoding="utf-8")
print(sql)
print(f"\nGuardado en {CARPETA_MODELOS}")
